<a href="https://colab.research.google.com/github/nivedita-rmsh/CLED/blob/main/CLED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import json
from collections import Counter

Verifying existence of MAVEN files

In [3]:
maven_dir = '/content/drive/MyDrive/NLP_data'
for f in os.listdir(maven_dir):
    size = os.path.getsize(os.path.join(maven_dir, f)) / 1024
    print(f"{f:40s} {size:.1f} KB")

test.jsonl                               15380.6 KB
valid.jsonl                              14807.4 KB
train.jsonl                              60456.8 KB
README.md                                2.9 KB


Load and Inspect Structure of JSON files

In [4]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(f'{maven_dir}/train.jsonl')

# Look at one document
doc = train[0]
print("Keys in a document:", list(doc.keys()))
print("Title:", doc['title'])
print("Number of sentences:", len(doc['content']))
print("Number of events:", len(doc['events']))

Keys in a document: ['title', 'id', 'content', 'events', 'negative_triggers']
Title: 2006 Pangandaran earthquake and tsunami
Number of sentences: 9
Number of events: 39


Looking into sentences with their event triggers highlighted.

In [5]:
for doc in train[:3]:
    print(f"\n=== {doc['title']} ===")
    for event in doc['events'][:2]:
        etype = event['type']
        for mention in event['mention'][:1]:
            sid   = mention['sent_id']
            start, end = mention['offset']
            tokens = doc['content'][sid]['tokens']
            trigger = ' '.join(tokens[start:end])
            sentence = ' '.join(tokens)
            print(f"  Event type : {etype}")
            print(f"  Trigger    : '{trigger}'  (tokens {start}–{end})")
            print(f"  Sentence   : {sentence}\n")


=== 2006 Pangandaran earthquake and tsunami ===
  Event type : Know
  Trigger    : 'observed'  (tokens 12–13)
  Sentence   : Several thousand kilometers to the southeast , surges of several meters were observed in northwestern Australia , but in Java the tsunami runups ( height above normal sea level ) were typically and resulted in the deaths of more than 600 people .

  Event type : Warning
  Trigger    : 'warning'  (tokens 27–28)
  Sentence   : since the shock was felt with only moderate intensity well inland , and even less so at the shore , the surge arrived with little or no warning .


=== Battle of Santa Clara (1927) ===
  Event type : Receiving
  Trigger    : 'received'  (tokens 2–3)
  Sentence   : The aircraft received fire from an enemy machine gun and a dive bombing raid ensued , with three bombs being dropped on the Nicaraguan rebels .

  Event type : Motion
  Trigger    : 'dropped'  (tokens 20–21)
  Sentence   : The aircraft received fire from an enemy machine gun and a 

Event Stats

In [6]:
event_types = Counter()
total_mentions = 0

for doc in train:
    for event in doc['events']:
        event_types[event['type']] += len(event['mention'])
        total_mentions += len(event['mention'])

print(f"Total documents   : {len(train)}")
print(f"Total event types : {len(event_types)}")
print(f"Total mentions    : {total_mentions}")
print(f"\nTop 10 event types:")
for etype, count in event_types.most_common(10):
    print(f"  {etype:30s} {count}")

Total documents   : 2913
Total event types : 168
Total mentions    : 77993

Top 10 event types:
  Catastrophe                    3145
  Attack                         2920
  Hostile_encounter              2856
  Causation                      2728
  Process_start                  2628
  Competition                    2460
  Motion                         2165
  Social_event                   1653
  Killing                        1625
  Conquering                     1432


Convert to BIO format

In [7]:
def doc_to_bio_examples(doc):
    examples = []
    trigger_map = {}
    for event in doc['events']:
        for mention in event['mention']:
            sid = mention['sent_id']
            trigger_map.setdefault(sid, []).append(
                (mention['offset'][0], mention['offset'][1], event['type'])
            )

    for sid, sent in enumerate(doc['content']):
        tokens = sent['tokens']
        labels = ['O'] * len(tokens)
        for start, end, etype in trigger_map.get(sid, []):
            for i in range(start, end):
                labels[i] = f"B-{etype}" if i == start else f"I-{etype}"
        examples.append({'tokens': tokens, 'labels': labels, 'doc_id': doc['id'], 'sent_id': sid})
    return examples

# Test it
sample = doc_to_bio_examples(train[0])
for ex in sample[:3]:
    pairs = list(zip(ex['tokens'], ex['labels']))
    non_o = [(t, l) for t, l in pairs if l != 'O']
    if non_o:
        print(ex['tokens'])
        print(ex['labels'])
        print()

['The', '2006', 'Pangandaran', 'earthquake', 'and', 'tsunami', 'occurred', 'on', 'July', '17', 'at', 'along', 'a', 'subduction', 'zone', 'off', 'the', 'coast', 'of', 'west', 'and', 'central', 'Java', ',', 'a', 'large', 'and', 'densely', 'populated', 'island', 'in', 'the', 'Indonesian', 'archipelago', '.']
['O', 'O', 'O', 'B-Catastrophe', 'O', 'B-Catastrophe', 'B-Presence', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['The', 'shock', 'had', 'a', 'moment', 'magnitude', 'of', '7.7', 'and', 'a', 'maximum', 'perceived', 'intensity', 'of', 'IV', '(', '``', 'Light', "''", ')', 'in', 'Jakarta', ',', 'the', 'capital', 'and', 'largest', 'city', 'of', 'Indonesia', '.']
['O', 'B-Catastrophe', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Know', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

['There', 'were', 'no', 'direct', 'effects', 'of', 'the', 'ear

Sanity Check

In [8]:
def check_bio_integrity(examples):
    errors = 0
    for ex in examples:
        prev = 'O'
        for i, label in enumerate(ex['labels']):
            if label.startswith('I-'):
                etype = label[2:]
                if prev != f'B-{etype}' and prev != f'I-{etype}':
                    print(f"BIO error at token {i}: '{label}' follows '{prev}'")
                    print(f"  Tokens: {ex['tokens']}")
                    errors += 1
            prev = label
    print(f"\nTotal BIO errors: {errors}")

check_bio_integrity(sample)


Total BIO errors: 0


Translation to FRENCH

In [9]:
!pip install transformers sentencepiece sacremoses -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 20.0 MB/s eta 0:00:00


In [10]:
from transformers import MarianMTModel, MarianTokenizer
import torch
import json
from tqdm import tqdm

In [11]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded successfully


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

def translate_batch(sentences, batch_size=32, max_length=128):
    """
    Translate a list of English strings to French.
    Returns a list of translated strings, same length as input.
    """
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        safe_batch = [s if s.strip() else "." for s in batch]

        inputs = tokenizer(
            safe_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            translated = model.generate(**inputs, max_length=max_length)

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [13]:

def translate_maven_split(input_path, output_path):
    """
    Reads a MAVEN .jsonl file, translates all sentence text to French,
    writes a new .jsonl with translated content alongside the original.
    """
    # Load all documents
    with open(input_path) as f:
        docs = [json.loads(line) for line in f]

    # Collect all sentences across all docs for batch translation
    # We track (doc_idx, sent_idx) so we can put them back
    all_sentences = []
    index_map     = []   # (doc_idx, sent_idx)

    for d_idx, doc in enumerate(docs):
        for s_idx, sent in enumerate(doc['content']):
            all_sentences.append(sent['sentence'])
            index_map.append((d_idx, s_idx))

    print(f"Translating {len(all_sentences)} sentences from {len(docs)} documents...")

    translated = []
    batch_size = 32
    for i in tqdm(range(0, len(all_sentences), batch_size)):
        batch = all_sentences[i : i + batch_size]
        translated.extend(translate_batch(batch, batch_size=batch_size))

    # Put translated sentences back into the document structure
    for (d_idx, s_idx), fr_sentence in zip(index_map, translated):
        docs[d_idx]['content'][s_idx]['sentence_fr'] = fr_sentence
        # Keep original sentence too — useful for debugging alignment

    with open(output_path, 'w') as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')

    print(f"Done. Saved to {output_path}")

In [14]:
maven_dir = '/content/drive/MyDrive/NLP_data'

translate_maven_split(
    f'{maven_dir}/train.jsonl',
    f'{maven_dir}/train_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/valid.jsonl',
    f'{maven_dir}/valid_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/test.jsonl',
    f'{maven_dir}/test_fr.jsonl'
)

Translating 32431 sentences from 2913 documents...


100%|██████████| 1014/1014 [30:43<00:00,  1.82s/it]


Done. Saved to /content/drive/MyDrive/NLP_data/train_fr.jsonl
Translating 8042 sentences from 710 documents...


100%|██████████| 252/252 [07:51<00:00,  1.87s/it]


Done. Saved to /content/drive/MyDrive/NLP_data/valid_fr.jsonl
Translating 9400 sentences from 857 documents...


100%|██████████| 294/294 [09:04<00:00,  1.85s/it]


Done. Saved to /content/drive/MyDrive/NLP_data/test_fr.jsonl


In [15]:
with open(f'{maven_dir}/train_fr.jsonl') as f:
    docs_fr = [json.loads(line) for line in f]

doc = docs_fr[0]
print(f"Document: {doc['title']}\n")

for sent in doc['content'][:4]:
    print(f"EN: {sent['sentence']}")
    print(f"FR: {sent['sentence_fr']}")
    print()

Document: 2006 Pangandaran earthquake and tsunami

EN: The 2006 Pangandaran earthquake and tsunami occurred on July 17 at along a subduction zone off the coast of west and central Java, a large and densely populated island in the Indonesian archipelago.
FR: Le tremblement de terre et le tsunami de Pangandaran de 2006 se sont produits le 17 juillet, le long d'une zone de subduction au large de la côte ouest et centrale de Java, une grande île densément peuplée de l'archipel indonésien.

EN: The shock had a moment magnitude of 7.7 and a maximum perceived intensity of IV ("Light") in Jakarta, the capital and largest city of Indonesia.
FR: Le choc avait une magnitude momentanée de 7,7 et une intensité maximale perçue de IV (« Lumière ») à Jakarta, la capitale et la plus grande ville d'Indonésie.

EN: There were no direct effects of the earthquake's shaking due to its low intensity, and the large loss of life from the event was due to the resulting tsunami, which inundated a portion of the 